# Module 4: Transformer Architecture & BERT — A Deep Dive

> **Series:** PyTorch Deep Dive → Transformer Architecture & BERT  
> **Prerequisites:** Modules 1, 2 & 3

---

## What you'll learn

Build and inspect every component of the Transformer encoder and BERT from the ground up.

| # | Section |
|---|---------|
| 1 | Environment setup & imports |
| 2 | Scaled dot-product attention from scratch |
| 3 | Multi-head attention implementation |
| 4 | Positional encoding visualisation |
| 5 | Transformer encoder block |
| 6 | Building a mini-BERT from scratch |
| 7 | Loading & inspecting pretrained BERT weights |
| 8 | Tokenisation deep dive |
| 9 | Embedding layer exploration |
| 10 | Attention weight visualisation |
| 11 | BERT for Masked Language Modelling (MLM) |
| 12 | BERT for Next Sentence Prediction (NSP) |
| 13 | Probing BERT layers with PCA / t-SNE |


---
## 1 · Environment Setup & Imports

Install required packages (run once):
```bash
pip install torch transformers matplotlib seaborn scikit-learn
```


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

print(f"PyTorch : {torch.__version__}")

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Device  : {device}")

# Seaborn style for all plots
sns.set_theme(style="whitegrid", palette="muted")


---
## 2 · Scaled Dot-Product Attention from Scratch

The core operation of every Transformer layer:

$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

- **Q** (Query): what we're looking for  
- **K** (Key): what each token advertises  
- **V** (Value): what each token contributes  
- **√d_k scaling**: prevents dot products from growing too large for softmax to handle (gradient saturation)


In [ ]:
def scaled_dot_product_attention(
    Q: torch.Tensor,                  # (B, T_q, d_k)
    K: torch.Tensor,                  # (B, T_k, d_k)
    V: torch.Tensor,                  # (B, T_k, d_v)
    mask: torch.Tensor | None = None, # (B, T_q, T_k) optional bool mask
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Returns:
        output       (B, T_q, d_v)  — weighted sum of values
        attn_weights (B, T_q, T_k)  — softmax attention scores
    """
    d_k = Q.size(-1)

    # Step 1: raw scores  (B, T_q, T_k)
    scores = torch.bmm(Q, K.transpose(1, 2)) / math.sqrt(d_k)

    # Step 2: mask (optional) — fill -inf so softmax gives 0 weight
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))

    # Step 3: softmax across the key dimension
    attn_weights = F.softmax(scores, dim=-1)

    # Step 4: weighted sum of values
    output = torch.bmm(attn_weights, V)

    return output, attn_weights


# --- Interactive demo ---
torch.manual_seed(0)
B, T, d_k, d_v = 1, 6, 16, 16
Q = torch.randn(B, T, d_k)
K = torch.randn(B, T, d_k)
V = torch.randn(B, T, d_v)

out, weights = scaled_dot_product_attention(Q, K, V)
print(f"Output shape       : {out.shape}")        # (1, 6, 16)
print(f"Attention weights  : {weights.shape}")    # (1, 6, 6)
print(f"Row sums (should=1): {weights[0].sum(dim=-1).tolist()}")

# Visualise the attention matrix
tokens = ["[CLS]", "the", "cat", "sat", "down", "[SEP]"]
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    weights[0].detach().numpy(),
    xticklabels=tokens, yticklabels=tokens,
    annot=True, fmt=".2f", cmap="Blues", ax=ax,
    linewidths=0.5, linecolor="white",
)
ax.set_title("Scaled Dot-Product Attention Weights")
ax.set_xlabel("Key (attending to)"); ax.set_ylabel("Query (from)")
plt.tight_layout(); plt.show()


---
## 3 · Multi-Head Attention

Instead of one attention function over full d_model dimensions, we run **h smaller attention heads in parallel**:

$$\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1,\ldots,\text{head}_h)\,W^O$$
$$\text{head}_i = \text{Attention}(QW_i^Q,\, KW_i^K,\, VW_i^V)$$

Each head learns to attend to different aspects (syntax, coreference, semantics…).  
In BERT-base: `d_model=768`, `num_heads=12`, `d_k = d_v = 64`.


In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Multi-head self-attention.

    Args:
        d_model   : total embedding dimension (e.g. 768 for BERT-base)
        num_heads : number of parallel attention heads (e.g. 12)
    """

    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.h  = num_heads
        self.d_k = d_model // num_heads   # dimension per head

        # Learned projections for Q, K, V and the output
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def _split_heads(self, x: torch.Tensor) -> torch.Tensor:
        """(B, T, d_model) → (B*h, T, d_k)"""
        B, T, _ = x.shape
        x = x.view(B, T, self.h, self.d_k)      # (B, T, h, d_k)
        x = x.permute(0, 2, 1, 3)               # (B, h, T, d_k)
        return x.reshape(B * self.h, T, self.d_k)

    def forward(
        self,
        x: torch.Tensor,                  # (B, T, d_model)
        mask: torch.Tensor | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        B, T, _ = x.shape

        # Project → split into heads
        Q = self._split_heads(self.W_q(x))   # (B*h, T, d_k)
        K = self._split_heads(self.W_k(x))
        V = self._split_heads(self.W_v(x))

        # Expand mask across heads if provided
        if mask is not None:
            mask = mask.repeat(self.h, 1, 1)  # (B*h, T, T)

        # Attention
        attn_out, attn_w = scaled_dot_product_attention(Q, K, V, mask)
        # attn_out : (B*h, T, d_k)

        # Merge heads: (B*h, T, d_k) → (B, T, d_model)
        attn_out = attn_out.view(B, self.h, T, self.d_k)
        attn_out = attn_out.permute(0, 2, 1, 3).reshape(B, T, self.h * self.d_k)

        return self.W_o(attn_out), attn_w.view(B, self.h, T, T)


# --- Shape verification ---
torch.manual_seed(42)
d_model, num_heads, seq_len, batch = 768, 12, 16, 2
mha = MultiHeadAttention(d_model, num_heads)
x   = torch.randn(batch, seq_len, d_model)

out, weights = mha(x)
print(f"Input  shape : {x.shape}")           # (2, 16, 768)
print(f"Output shape : {out.shape}")          # (2, 16, 768)
print(f"Weight shape : {weights.shape}")      # (2, 12, 16, 16)  — B, heads, T, T
print(f"Row sums ≈ 1 : {weights[0, 0].sum(dim=-1)[:4].tolist()}")


---
## 4 · Positional Encoding Visualisation

Transformers have no built-in sense of order — we inject position information as a fixed sinusoidal signal added to the token embeddings:

$$PE_{(pos,\,2i)}   = \sin\!\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$
$$PE_{(pos,\,2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$

Each position gets a unique "fingerprint" across dimensions. The sinusoidal choice means the model can **generalise to longer sequences** than seen during training (relative offsets have a fixed pattern regardless of absolute position).


In [ ]:
def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Returns a (max_len, d_model) positional encoding matrix."""
    pe  = torch.zeros(max_len, d_model)
    pos = torch.arange(max_len).unsqueeze(1).float()           # (max_len, 1)
    dim = torch.arange(0, d_model, 2).float()                  # even indices
    div = torch.pow(10000.0, dim / d_model)                    # frequency denominators

    pe[:, 0::2] = torch.sin(pos / div)   # even dimensions
    pe[:, 1::2] = torch.cos(pos / div)   # odd dimensions
    return pe


pe = sinusoidal_positional_encoding(max_len=128, d_model=128)
print(f"PE shape: {pe.shape}")   # (128, 128)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Heatmap of the full encoding matrix
sns.heatmap(pe.numpy(), cmap="RdBu_r", center=0, ax=axes[0],
            xticklabels=16, yticklabels=16)
axes[0].set_title("Sinusoidal Positional Encoding Matrix")
axes[0].set_xlabel("Embedding dimension"); axes[0].set_ylabel("Position")

# Line plot: how 4 specific dimensions oscillate across positions
for dim_idx in [0, 10, 30, 64]:
    axes[1].plot(pe[:64, dim_idx].numpy(), label=f"dim {dim_idx}")
axes[1].set_title("PE values across positions (selected dims)")
axes[1].set_xlabel("Position"); axes[1].set_ylabel("Value")
axes[1].legend(fontsize=8)

plt.tight_layout(); plt.show()


---
## 5 · Transformer Encoder Block

One encoder layer stacks four sub-modules:

```
x ──► MultiHeadAttention ──► Add & LayerNorm ──► FeedForward ──► Add & LayerNorm ──► x'
         ↑___________________________↑                 ↑_________________________↑
              residual connection                          residual connection
```

The **Feed-Forward Network** is two linear layers with GELU activation:
$$\text{FFN}(x) = \text{GELU}(x W_1 + b_1)\,W_2 + b_2$$

`d_ff` is typically 4× `d_model` (3072 in BERT-base). Residual connections + LayerNorm stabilise training in deep networks.


In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class TransformerEncoderBlock(nn.Module):
    """
    One Transformer encoder layer:
        x → MHA → Add+LN → FFN → Add+LN → x'
    """

    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.attn   = MultiHeadAttention(d_model, num_heads)
        self.ff     = FeedForward(d_model, d_ff, dropout)
        self.norm1  = nn.LayerNorm(d_model)
        self.norm2  = nn.LayerNorm(d_model)
        self.drop   = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, mask: torch.Tensor | None = None):
        # Sub-layer 1: self-attention + residual
        attn_out, _ = self.attn(x, mask)
        x = self.norm1(x + self.drop(attn_out))   # Add & LayerNorm

        # Sub-layer 2: feed-forward + residual
        x = self.norm2(x + self.ff(x))             # Add & LayerNorm
        return x


# --- Shape test (BERT-base dimensions) ---
enc_block = TransformerEncoderBlock(d_model=768, num_heads=12, d_ff=3072)
x = torch.randn(2, 16, 768)   # (batch=2, seq_len=16, d_model=768)
out = enc_block(x)
print(f"Input  shape : {x.shape}")
print(f"Output shape : {out.shape}")   # must match input
n_params = sum(p.numel() for p in enc_block.parameters())
print(f"Params per block: {n_params:,}  (~{n_params/1e6:.1f}M)")
# BERT-base has 12 such blocks → 12 × ~7.1M ≈ 85M of the 110M total


---
## 6 · Building a Mini-BERT from Scratch

BERT adds three embedding tables on top of the Transformer encoder:

| Embedding | Purpose |
|-----------|---------|
| **Token** | Maps each vocabulary token ID → vector |
| **Position** | Learned (not sinusoidal!) position signals |
| **Segment** | Distinguishes sentence A vs sentence B (0 or 1) |

The final input is the **sum** of all three. We then stack N encoder blocks.


In [ ]:
class MiniBERT(nn.Module):
    """
    Minimal BERT-like encoder (no pre-training head).

    Demonstrates token + position + segment embeddings
    followed by N stacked TransformerEncoderBlocks.
    """

    def __init__(
        self,
        vocab_size: int  = 30_522,   # bert-base-uncased vocab
        d_model:    int  = 768,
        num_heads:  int  = 12,
        num_layers: int  = 12,
        d_ff:       int  = 3072,
        max_len:    int  = 512,
        dropout:    float = 0.1,
    ):
        super().__init__()
        # Three embedding tables
        self.tok_emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len,    d_model)
        self.seg_emb = nn.Embedding(2,          d_model)   # segment 0 or 1

        self.emb_norm = nn.LayerNorm(d_model)
        self.emb_drop = nn.Dropout(dropout)

        # N encoder blocks
        self.layers = nn.ModuleList([
            TransformerEncoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

    def forward(
        self,
        input_ids:      torch.Tensor,              # (B, T)
        segment_ids:    torch.Tensor | None = None, # (B, T)
        attention_mask: torch.Tensor | None = None, # (B, T)  1=real token, 0=pad
    ) -> torch.Tensor:                              # (B, T, d_model)

        B, T = input_ids.shape
        positions   = torch.arange(T, device=input_ids.device).unsqueeze(0)  # (1, T)
        seg         = segment_ids if segment_ids is not None else torch.zeros_like(input_ids)

        # Sum three embeddings
        x = self.tok_emb(input_ids) + self.pos_emb(positions) + self.seg_emb(seg)
        x = self.emb_drop(self.emb_norm(x))

        # Build attention mask for MHA: (B, T, T)
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(1) * attention_mask.unsqueeze(2)  # (B, T, T)
        else:
            mask = None

        for layer in self.layers:
            x = layer(x, mask)

        return x   # (B, T, d_model) — contextualised token representations


# --- Instantiate and count parameters ---
mini_bert = MiniBERT()
total = sum(p.numel() for p in mini_bert.parameters())
print(f"MiniBERT total params : {total:,}  (~{total/1e6:.0f}M)")
print("bert-base-uncased     : 109,482,240  (~110M)")
print()

# Run a toy batch
input_ids = torch.randint(0, 30522, (2, 32))   # batch=2, seq_len=32
hidden    = mini_bert(input_ids)
print(f"Output hidden states  : {hidden.shape}")  # (2, 32, 768)


---
## 7 · Loading & Inspecting Pretrained BERT Weights

Let's load the real `bert-base-uncased` and explore its internal structure: how many parameters live in each component, and what shapes the weight matrices are.


In [ ]:
from transformers import AutoModel

bert = AutoModel.from_pretrained("bert-base-uncased")
bert.eval()

# Collect param counts by component group
groups = {"embeddings": 0, "encoder": 0, "pooler": 0}
for name, param in bert.named_parameters():
    top = name.split(".")[0]          # "embeddings", "encoder", "pooler"
    groups[top] = groups.get(top, 0) + param.numel()

total = sum(groups.values())
print(f"{'Component':<15} {'Params':>12}  {'%':>6}")
print("─" * 36)
for k, v in groups.items():
    print(f"{k:<15} {v:>12,}  {v/total*100:>5.1f}%")
print("─" * 36)
print(f"{'TOTAL':<15} {total:>12,}  100.0%")

# Bar chart
fig, ax = plt.subplots(figsize=(6, 3))
ax.barh(list(groups.keys()), [v / 1e6 for v in groups.values()], color=["#4C72B0", "#DD8452", "#55A868"])
ax.set_xlabel("Parameters (millions)")
ax.set_title("BERT-base Parameter Distribution by Component")
for i, v in enumerate(groups.values()):
    ax.text(v / 1e6 + 0.3, i, f"{v/1e6:.1f}M", va="center")
plt.tight_layout(); plt.show()


---
## 8 · Tokenisation Deep Dive

BERT uses **WordPiece** tokenisation — unknown words are split into the longest known subword prefixes, with continuation pieces prefixed by `##`.

Special tokens:
| Token | ID | Role |
|-------|----|------|
| `[CLS]` | 101 | Sentence-level representation (always first) |
| `[SEP]` | 102 | Sentence separator / end marker |
| `[PAD]` | 0   | Padding to fixed length |
| `[MASK]`| 103 | Replaced during MLM pre-training |
| `[UNK]` | 100 | Truly unknown token (rare with WordPiece) |


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Compare tokenisation across word types
test_words = [
    "running",          # common — likely one token
    "uncharacteristically",  # long — will be split
    "ChatGPT",          # OOV proper noun
    "PyTorch",          # technical term
    "supercalifragilistic",  # nonsense long word
    "café",             # accented character
]

print(f"{'Word':<28} {'Tokens':<45} IDs")
print("─" * 90)
for word in test_words:
    tokens = tokenizer.tokenize(word)
    ids    = tokenizer.convert_tokens_to_ids(tokens)
    print(f"{word:<28} {str(tokens):<45} {ids}")

# Special token IDs
print("\nSpecial tokens:")
for name in ["[CLS]", "[SEP]", "[PAD]", "[MASK]", "[UNK]"]:
    print(f"  {name:8s} → ID {tokenizer.convert_tokens_to_ids(name)}")


---
## 9 · Embedding Layer Exploration

BERT's input embedding is the **sum** of three independent lookup tables:

```
input_embedding = token_embedding + position_embedding + segment_embedding
```

This additive design means the model learns to disentangle identity, order, and sentence membership independently. We can extract and inspect each table directly.


In [ ]:
emb = bert.embeddings   # shorthand

tok_weight = emb.word_embeddings.weight.detach()        # (30522, 768)
pos_weight = emb.position_embeddings.weight.detach()    # (512, 768)
seg_weight = emb.token_type_embeddings.weight.detach()  # (2, 768)

print(f"Token embeddings  : {tok_weight.shape}  — {tok_weight.shape[0]:,} vocab × 768 dims")
print(f"Position embeddings: {pos_weight.shape}")
print(f"Segment embeddings: {seg_weight.shape}   — segment 0 and segment 1")

# L2 norm of token embeddings for a set of words
words = ["the", "cat", "sat", "supercalifragilistic",
         "bank", "river", "machine", "learning", "transformer", "attention"]
norms = []
for w in words:
    ids = tokenizer.convert_tokens_to_ids(tokenizer.tokenize(w))
    # Average embedding across subword tokens
    vecs  = tok_weight[ids]
    norms.append(vecs.norm(dim=-1).mean().item())

fig, ax = plt.subplots(figsize=(8, 3))
bars = ax.bar(words, norms, color=sns.color_palette("muted", len(words)))
ax.set_ylabel("Mean L2 norm of token embedding")
ax.set_title("Token Embedding L2 Norms: common vs rare words")
ax.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

# Verify additivity: embed a real sentence manually vs via BERT
sentence = "Hello world"
enc = tokenizer(sentence, return_tensors="pt")
ids  = enc["input_ids"]       # (1, T)
segs = enc["token_type_ids"]  # (1, T)
pos  = torch.arange(ids.shape[1]).unsqueeze(0)

manual = (
    emb.word_embeddings(ids)
    + emb.position_embeddings(pos)
    + emb.token_type_embeddings(segs)
)
manual = emb.LayerNorm(manual)  # BERT also applies LayerNorm

with torch.no_grad():
    bert_emb = emb(ids)

print(f"\nManual vs BERT embedding max diff: {(manual - bert_emb).abs().max().item():.2e}  ← should be ≈ 0")


---
## 10 · Attention Weight Visualisation

BERT has **12 layers × 12 heads = 144 attention patterns**. Different heads learn different linguistic relationships:

- Some heads attend to the next/previous token (local syntax)
- Some heads attend from `[SEP]` to all tokens (global gathering)
- Some heads track coreference or long-range dependencies

Pass `output_attentions=True` to get all 144 patterns back.


In [ ]:
sentence = "The bank can guarantee deposits will eventually cover future tuition costs."
enc = tokenizer(sentence, return_tensors="pt")
tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])

with torch.no_grad():
    outputs = bert(**enc, output_attentions=True)

# outputs.attentions is a tuple of 12 tensors, each (1, 12, T, T)
all_attn = torch.stack(outputs.attentions)   # (12, 1, 12, T, T)
all_attn = all_attn.squeeze(1)               # (12, 12, T, T)

print(f"Attention tensor: {all_attn.shape}  — (layers, heads, seq, seq)")

# Plot all 12 heads from layer 5 (0-indexed)
LAYER = 5
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for head in range(12):
    w = all_attn[LAYER, head].numpy()
    sns.heatmap(
        w, ax=axes[head],
        xticklabels=tokens, yticklabels=tokens,
        cmap="Blues", cbar=False,
        linewidths=0.3, linecolor="white",
    )
    axes[head].set_title(f"Layer {LAYER+1} · Head {head+1}", fontsize=9)
    axes[head].tick_params(axis='both', labelsize=6, rotation=45)

plt.suptitle(f"All 12 attention heads — Layer {LAYER+1}", fontsize=13, y=1.01)
plt.tight_layout(); plt.show()


---
## 11 · BERT for Masked Language Modelling (MLM)

During pre-training, BERT randomly replaces 15% of tokens with `[MASK]` and learns to predict the original token. This forces it to build deep bidirectional context representations.

We can use this directly to fill in blanks — or to probe what BERT "thinks" belongs in a given position.


In [ ]:
from transformers import BertForMaskedLM

mlm_model = BertForMaskedLM.from_pretrained("bert-base-uncased")
mlm_model.eval()

def fill_mask(sentence: str, top_k: int = 5) -> None:
    """Run MLM on a sentence with one [MASK] token and print top-k predictions."""
    enc         = tokenizer(sentence, return_tensors="pt")
    mask_idx    = (enc["input_ids"] == tokenizer.mask_token_id).nonzero(as_tuple=True)[1]

    with torch.no_grad():
        logits  = mlm_model(**enc).logits          # (1, T, vocab_size)

    probs       = logits[0, mask_idx, :].softmax(dim=-1)
    top_probs, top_ids = probs.topk(top_k, dim=-1)

    tokens_str = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
    print(f"\nInput  : {' '.join(tokens_str)}")
    print(f"{'Rank':<5} {'Token':<20} {'Probability':>12}")
    print("─" * 40)
    for rank, (prob, tok_id) in enumerate(zip(top_probs[0], top_ids[0]), 1):
        word = tokenizer.convert_ids_to_tokens(tok_id.item())
        print(f"{rank:<5} {word:<20} {prob.item():>12.4f}")


# Try different mask positions and word types
fill_mask("The [MASK] can guarantee deposits will eventually cover costs.")
fill_mask("The scientist [MASK] a new experiment in the laboratory.")
fill_mask("Paris is the [MASK] of France.")


---
## 12 · BERT for Next Sentence Prediction (NSP)

The second pre-training objective teaches BERT whether sentence B **naturally follows** sentence A. The `[CLS]` token's representation is passed through a binary classifier:

- **IsNext** (label 0): sentence B genuinely follows A
- **NotNext** (label 1): sentence B is a random sentence

This gives BERT its ability to understand cross-sentence relationships, useful for QA and NLI tasks.


In [ ]:
from transformers import BertForNextSentencePrediction

nsp_model = BertForNextSentencePrediction.from_pretrained("bert-base-uncased")
nsp_model.eval()

def nsp_score(sentence_a: str, sentence_b: str) -> dict:
    """Returns IsNext / NotNext logits and probabilities for a sentence pair."""
    enc = tokenizer(sentence_a, sentence_b, return_tensors="pt")
    with torch.no_grad():
        logits = nsp_model(**enc).logits   # (1, 2)
    probs  = logits.softmax(dim=-1)[0]
    return {
        "IsNext":  probs[0].item(),
        "NotNext": probs[1].item(),
    }


# Sentence pairs to test
pairs = [
    ("The dog chased the ball across the garden.", "It rolled under the fence and disappeared."),
    ("The dog chased the ball across the garden.", "The Eiffel Tower is located in Paris, France."),
    ("She opened the letter carefully.", "Inside was a handwritten note from her sister."),
    ("She opened the letter carefully.", "Quantum mechanics describes subatomic phenomena."),
]

labels   = ["(consecutive)", "(random)", "(consecutive)", "(random)"]
is_nexts = []
not_nexts = []
pair_labels = []

for (a, b), label in zip(pairs, labels):
    scores = nsp_score(a, b)
    is_nexts.append(scores["IsNext"])
    not_nexts.append(scores["NotNext"])
    pair_labels.append(f"Pair {pairs.index((a,b))+1} {label}")
    print(f"A: {a[:55]}...")
    print(f"B: {b[:55]}...")
    print(f"   IsNext={scores['IsNext']:.3f}  NotNext={scores['NotNext']:.3f}\n")

# Bar chart
x = np.arange(len(pairs))
width = 0.35
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - width/2, is_nexts,  width, label="IsNext",  color="#4C72B0")
ax.bar(x + width/2, not_nexts, width, label="NotNext", color="#DD8452")
ax.set_xticks(x); ax.set_xticklabels(pair_labels, rotation=10, fontsize=8)
ax.set_ylabel("Probability"); ax.set_title("NSP: IsNext vs NotNext Scores")
ax.legend(); plt.tight_layout(); plt.show()


---
## 13 · Probing BERT Layer Representations with PCA / t-SNE

Lower BERT layers encode **surface/syntactic** features; upper layers encode **semantic/task** features. We can visualise this progression by extracting hidden states from every layer for a set of sentences grouped by semantic category, then projecting to 2D with PCA and t-SNE.


In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Sentences from 4 semantic categories
categories = {
    "Animals":    ["The cat sat on the mat.", "Dogs love to play fetch.",
                   "The eagle soared above the mountains.", "A dolphin leapt from the ocean."],
    "Technology": ["The GPU accelerates deep learning training.", "BERT uses the Transformer architecture.",
                   "Neural networks learn from data.", "The CPU processes billions of instructions per second."],
    "Food":       ["She baked a chocolate cake for the party.", "The pasta was perfectly al dente.",
                   "Fresh tomatoes make the best salsa.", "He ordered a double espresso at the café."],
    "Sports":     ["The striker scored in the final minute.", "She won the marathon by two seconds.",
                   "The tennis player served an ace.", "Basketball requires speed and coordination."],
}

all_sentences = [s for sents in categories.values() for s in sents]
all_labels    = [cat for cat, sents in categories.items() for _ in sents]

# Extract [CLS] token hidden states from every layer
def get_cls_hidden_states(sentences: list[str]) -> np.ndarray:
    """Returns array of shape (num_layers+1, N, 768) — layer 0 is the embedding layer."""
    enc = tokenizer(sentences, return_tensors="pt", padding=True, truncation=True, max_length=64)
    with torch.no_grad():
        out = bert(**enc, output_hidden_states=True)
    # out.hidden_states: tuple of (N_layers+1) tensors, each (B, T, 768)
    cls_per_layer = np.stack([h[:, 0, :].numpy() for h in out.hidden_states])  # (13, B, 768)
    return cls_per_layer


hidden = get_cls_hidden_states(all_sentences)  # (13, 16, 768)
print(f"Hidden states shape: {hidden.shape}  — (layers, sentences, dim)")

# Plot PCA projections at layer 1, 6, and 12
palette = {"Animals": "#4C72B0", "Technology": "#DD8452", "Food": "#55A868", "Sports": "#C44E52"}
plot_layers = [1, 6, 12]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, layer_idx in zip(axes, plot_layers):
    X = hidden[layer_idx]                  # (16, 768)
    pca = PCA(n_components=2, random_state=0)
    X2  = pca.fit_transform(X)             # (16, 2)

    for i, (x, label) in enumerate(zip(X2, all_labels)):
        ax.scatter(x[0], x[1], color=palette[label],
                   s=80, edgecolors="white", linewidths=0.5, zorder=3)
        ax.annotate(f"{i+1}", (x[0], x[1]), fontsize=6, ha="center", va="center")

    # Legend
    for cat, col in palette.items():
        ax.scatter([], [], color=col, label=cat, s=60)
    ax.legend(fontsize=7, loc="upper right")
    var = pca.explained_variance_ratio_.sum() * 100
    ax.set_title(f"Layer {layer_idx}  (PCA, {var:.0f}% var)")
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")

plt.suptitle("BERT [CLS] Representations: PCA by Layer", fontsize=13)
plt.tight_layout(); plt.show()


In [ ]:
# t-SNE on the final layer (layer 12) for richer comparison
X_final = hidden[12]   # (16, 768)

# PCA → 50 dims first (standard practice before t-SNE)
X_pca50 = PCA(n_components=min(16, X_final.shape[1]), random_state=0).fit_transform(X_final)

tsne    = TSNE(n_components=2, perplexity=5, random_state=0, n_iter=1000)
X_tsne  = tsne.fit_transform(X_pca50)

fig, ax = plt.subplots(figsize=(7, 6))
for i, (x, label, sent) in enumerate(zip(X_tsne, all_labels, all_sentences)):
    ax.scatter(x[0], x[1], color=palette[label], s=100,
               edgecolors="white", linewidths=0.7, zorder=3)
    ax.annotate(sent[:25] + "…", (x[0], x[1]),
                fontsize=6, xytext=(4, 4), textcoords="offset points")

for cat, col in palette.items():
    ax.scatter([], [], color=col, label=cat, s=70)
ax.legend(fontsize=9, loc="best")
ax.set_title("t-SNE of BERT Layer 12 [CLS] Representations")
ax.set_xlabel("t-SNE 1"); ax.set_ylabel("t-SNE 2")
plt.tight_layout(); plt.show()


---
## Summary

You've now built and inspected every layer of the Transformer / BERT stack from first principles:

| Section | What you built / explored |
|---------|--------------------------|
| 2 | Scaled dot-product attention — the atomic operation |
| 3 | Multi-head attention — parallel attention heads |
| 4 | Sinusoidal positional encoding — injecting order |
| 5 | Encoder block — attention + FFN + residuals + LayerNorm |
| 6 | Mini-BERT — full model with three embedding tables |
| 7 | Pretrained BERT weights — parameter distribution |
| 8 | WordPiece tokenisation — subword splits & special tokens |
| 9 | Embedding layers — token, position, segment & their sum |
| 10 | Attention visualisation — 144 patterns, linguistic roles |
| 11 | MLM — fill-mask predictions with probabilities |
| 12 | NSP — sentence-pair coherence scoring |
| 13 | PCA / t-SNE probing — semantic clustering across layers |

### Ideas to explore next

1. **Layer-wise probing** — train a linear classifier on each layer's `[CLS]` token for a sentiment task. Which layer gives the best accuracy?
2. **Attention head pruning** — zero out individual heads; which heads can be removed with minimal performance loss?
3. **BERT vs RoBERTa** — swap `bert-base-uncased` for `roberta-base` (no NSP, more data). Do the attention patterns differ?
4. **Fine-tuning probe** — after fine-tuning on a downstream task (Module 3), re-run the PCA probe. Do the clusters sharpen?
